# RQ1.2 — детекторы RT-DETR на DIVA-HisDB и U-DIADS-TL

Переобучает всю сетку RQ1.2 по рецепту, собранному за 24-25 августа. Каждый пункт
ниже подтверждён измерением, ссылки на числа — в
`99_evaluation/summaries/diagnostics/DIVA_detector_findings.md`.

## Рецепт

**Обучение**: 50 эпох, lr 1e-4, батч 1, **накопление 2**, планировщик constant,
seed 42, bf16, разрешение 1152, **EMA выключена**.

- *Накопление 2, а не 16.* Старые арки U-DIADS обучались при 16, и это давало
  вшестеро меньше шагов оптимизатора: медиана test mAP50 0.20-0.42 против 0.75-0.90
  на DIVA. После смены: 0.877 / 0.856 / 0.868 на трёх подмножествах U-DIADS.
- *EMA выключена.* Decay 0.993 усредняет по ~140 шагам, а эпоха на малых данных
  это 10 шагов, поэтому тень тянет в модель необученные состояния. Оба прогона,
  где тренер выбрал усреднённые веса, обвалились (Line_IU 0.400 и 0.620) при
  0.72-0.92 у всех прогонов с сырыми весами.
- *50 эпох.* На CS18 это даёт тот же результат, что 600 (Line_IU 0.916), при
  бюджете в двенадцать раз меньше.

**Разметка**: на DIVA — `coco_task2_<subset>/`, собранная из TASK-2, той самой,
по которой считаются метрики. Обычная COCO размечает межстрочные глоссы как
строки: на CS18 это 170 объектов на страницу против 27 целевых, и детектор тратит
ёмкость на то, за что потом штрафуется. Переход дал CS18 +0.15 по Line_IU.
На U-DIADS такого расхождения нет — там пиксельные маски строк уже целевые.

**Постобработка** (применяется при оценке, не здесь): `close-fraction=0` —
морфологическое замыкание по умолчанию 0.04 сваривает соседние строки и стоит
0.24 FM на Syr341, а при 0.10 обваливает результат в ноль.

## 1. Конфигурация

In [1]:

import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL    = "https://github.com/Maserezio/fsl-tl-manuscripts.git"
REPO_BRANCH = "main"
DRIVE_ROOT  = Path("/content/drive/MyDrive/Thesis")
REPO_DIR    = Path("/content/repo")
WORK_DIR    = Path("/content/work")

DRIVE_DATA   = DRIVE_ROOT / "00_data"
DRIVE_MODELS = DRIVE_ROOT / "80_models"
DRIVE_OUT    = DRIVE_ROOT / "99_evaluation"
LOCAL_DATA   = WORK_DIR / "00_data"
LOCAL_MODELS = WORK_DIR / "80_models"

IMAGE_SIZE, EPOCHS = 1152, 50
TRANSFORMERS_PIN = "5.14.1"

DIVA   = ["CB55", "CS18", "CS863"]
UDIADS = ["Latin14396", "Latin2", "Syr341"]

# (backbone, init). DINOv3 exists only for convnext_tiny -- the smallest DINO weights
# of any family are 21.6M, so XS/S cannot have that column at all.
CONV_PVT = [("convnext_femto", "random"), ("convnext_femto", "imagenet"),
            ("convnext_pico",  "random"), ("convnext_pico",  "imagenet"),
            ("convnext_tiny",  "random"), ("convnext_tiny",  "imagenet"),
            ("convnext_tiny",  "dinov3"),
            ("pvt_v2_b0", "random"), ("pvt_v2_b0", "imagenet"),
            ("pvt_v2_b1", "random"), ("pvt_v2_b1", "imagenet"),
            ("pvt_v2_b2", "random"), ("pvt_v2_b2", "imagenet"),
            ("", "random")]                      # "" = stock R50-vd
# Flat ViTs reach RT-DETR only through hier_encoder's SFP neck: fed directly they
# emit every level at one stride and the model dies on the concat.
VIT = [("vit_tiny",  "random"), ("vit_tiny",  "imagenet"),
       ("vit_small", "random"), ("vit_small", "imagenet"), ("vit_small", "dinov3")]

GRID = CONV_PVT + VIT

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def sh(cmd, check=True, cwd=None, env=None):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd,
                          env={**os.environ, **(env or {})})


ARMS = ([("DIVA-HisDB", s, b, i) for s in DIVA for b, i in GRID]
        + [("U-DIADS-TL", s, b, i) for s in UDIADS for b, i in GRID])
print(f"конфигураций на подмножество: {len(GRID)}")
print(f"всего детекторов: {len(ARMS)} "
      f"({len(DIVA) * len(GRID)} DIVA + {len(UDIADS) * len(GRID)} U-DIADS)")

конфигураций на подмножество: 19
всего детекторов: 114 (57 DIVA + 57 U-DIADS)


## 2. Диск

In [2]:

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} не найден — создайте его на Диске"
assert DRIVE_DATA.is_dir(), f"{DRIVE_DATA} не найден — залейте туда 00_data"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("диск в порядке:", DRIVE_ROOT)

Mounted at /content/drive
диск в порядке: /content/drive/MyDrive/Thesis


## 3. Репозиторий

Из клона берутся только отслеживаемые файлы `50_modelling/instance_segmentation/rtdetr_bbox_unet/`. Гитигнорные
`00_data` и `80_models` подставляются симлинками на локальный SSD — обучение
с FUSE-монтирования Диска идёт в разы медленнее.

In [3]:

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

for d in (LOCAL_DATA, LOCAL_MODELS):
    d.mkdir(parents=True, exist_ok=True)
for name, target in (("00_data", LOCAL_DATA), ("80_models", LOCAL_MODELS)):
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target)

STAGE_DIR = REPO_DIR / "50_modelling" / "02_2stage"
assert (STAGE_DIR / "train_rtdetr_hf.py").exists(), "train_rtdetr_hf.py не найден"
print("репозиторий готов:", REPO_DIR)

$ git clone --depth 1 --branch main https://github.com/Maserezio/fsl-tl-manuscripts.git /content/repo
репозиторий готов: /content/repo


## 4. Данные

DIVA обучается на `coco_task2_<subset>/`. Если этого каталога нет на Диске,
следующая ячейка соберёт его из TASK-2 разметки скриптом `00_data/scripts/task2_to_coco.py`.

In [4]:

def resolve(root, rel):
    """Locate rel under root, tolerating case differences in the subset segment.

    The U-DIADS trees are lowercase on this machine (coco_dataset_latin14396) and the
    trainer builds the same, but a Drive copy may well be capitalised -- matching the
    literal string only would fail there with a misleading "not on Drive".
    """
    direct = root / rel
    if direct.exists():
        return direct
    node = root
    for part in Path(rel).parts:
        if (node / part).exists():
            node = node / part
            continue
        hit = next((c for c in node.iterdir() if c.name.lower() == part.lower()), None)             if node.is_dir() else None
        if hit is None:
            return None
        node = hit
    return node


NEEDED = []
for s in DIVA:
    NEEDED += [f"DIVA-HisDB/yolo_dataset_{s}/images",
               f"DIVA-HisDB/{s}/PAGE-gt-{s}-TASK-2/TASK-2",
               f"DIVA-HisDB/{s}/img-{s}/img"]
for s in UDIADS:
    NEEDED += [f"U-DIADS-TL/coco_dataset_{s.lower()}",
               f"U-DIADS-TL/yolo_dataset_{s.lower()}/images"]

missing = []
for rel in NEEDED:
    dst = LOCAL_DATA / rel
    if dst.exists():
        continue
    src = resolve(DRIVE_DATA, rel)
    if src is None:
        missing.append(rel)
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print(f"[copy] {src.name} -> {rel}")
assert not missing, f"нет на Диске: {missing[:6]}"

# TASK-2 COCO for DIVA: rebuild locally when the Drive copy is absent.
for s in DIVA:
    out = LOCAL_DATA / f"DIVA-HisDB/coco_task2_{s}"
    if out.exists():
        continue
    src = resolve(DRIVE_DATA, f"DIVA-HisDB/coco_task2_{s}")
    if src is not None:
        shutil.copytree(src, out)
        print(f"[copy] coco_task2_{s}")
    else:
        sh(f"python 00_data/scripts/task2_to_coco.py --subsets {s} "
           f"--data-root {LOCAL_DATA}/DIVA-HisDB", cwd=REPO_DIR)
print("данные на месте")

[copy] images -> DIVA-HisDB/yolo_dataset_CB55/images
[copy] TASK-2 -> DIVA-HisDB/CB55/PAGE-gt-CB55-TASK-2/TASK-2
[copy] img -> DIVA-HisDB/CB55/img-CB55/img
[copy] images -> DIVA-HisDB/yolo_dataset_CS18/images
[copy] TASK-2 -> DIVA-HisDB/CS18/PAGE-gt-CS18-TASK-2/TASK-2
[copy] img -> DIVA-HisDB/CS18/img-CS18/img
[copy] images -> DIVA-HisDB/yolo_dataset_CS863/images
[copy] TASK-2 -> DIVA-HisDB/CS863/PAGE-gt-CS863-TASK-2/TASK-2
[copy] img -> DIVA-HisDB/CS863/img-CS863/img
[copy] coco_dataset_Latin14396 -> U-DIADS-TL/coco_dataset_latin14396
[copy] images -> U-DIADS-TL/yolo_dataset_latin14396/images
[copy] coco_dataset_Latin2 -> U-DIADS-TL/coco_dataset_latin2
[copy] images -> U-DIADS-TL/yolo_dataset_latin2/images
[copy] coco_dataset_Syr341 -> U-DIADS-TL/coco_dataset_syr341
[copy] images -> U-DIADS-TL/yolo_dataset_syr341/images
$ python 00_data/scripts/task2_to_coco.py --subsets CB55 --data-root /content/work/00_data/DIVA-HisDB


CalledProcessError: Command 'python 00_data/scripts/task2_to_coco.py --subsets CB55 --data-root /content/work/00_data/DIVA-HisDB' returned non-zero exit status 2.

## 5. Зависимости

In [ ]:

sh(f'pip -q install "transformers=={TRANSFORMERS_PIN}" torchmetrics '
   f'albumentations pycocotools datasets timm')
sh("nvidia-smi || true", check=False)

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert transformers.__version__.startswith(TRANSFORMERS_PIN.split(".")[0]), \
    "мажорная версия transformers отличается от пина — перезапустите среду"
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"{props.total_memory / 1e9:.0f} ГБ")
print("bf16:", torch.cuda.is_bf16_supported(), "(на T4 False — упадёт в fp16)")

## 6. Проверка памяти

Самая тяжёлая арка сетки — `pvt_v2_b2` при 1152 px. Если не влезает, уменьшать
`IMAGE_SIZE` нельзя без потери сопоставимости: остальные арки считались при 1152.

In [ ]:

import gc
from transformers import AutoConfig, RTDetrConfig, AutoModelForObjectDetection

bc = AutoConfig.from_pretrained("OpenGVLab/pvt_v2_b2")
bc.out_indices = [1, 2, 3]                # stage2/3/4 -> страйды 8/16/32
cfg = RTDetrConfig.from_pretrained(
    "PekingU/rtdetr_r50vd", backbone_config=bc, use_timm_backbone=False,
    num_feature_levels=3, num_labels=1,
    id2label={0: "TextLine"}, label2id={"TextLine": 0})
model = AutoModelForObjectDetection.from_config(cfg).cuda().train()
try:
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device="cuda")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    with torch.autocast("cuda", dtype=dtype):
        loss = model(pixel_values=x, labels=[{
            "class_labels": torch.zeros(1, dtype=torch.long, device="cuda"),
            "boxes": torch.tensor([[.5, .5, .2, .05]], device="cuda")}]).loss
    loss.backward()
    print(f"OK при {IMAGE_SIZE}px: пик {torch.cuda.max_memory_allocated()/1e9:.1f} ГБ")
except torch.cuda.OutOfMemoryError:
    print(f"НЕ ВЛЕЗАЕТ при {IMAGE_SIZE}px — нужна карта побольше")
finally:
    del model
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

## 7. Обучение

Вызывается репозиторный `train_rtdetr_hf.py` через переменные окружения, поэтому
рецепт не дублируется. Готовая арка сразу уезжает на Диск, так что обрыв сессии
не теряет работу. Уже посчитанные арки пропускаются.

In [ ]:

def spec(dataset, subset, backbone, init):
    """Run name, output path and the environment the trainer reads."""
    name = f"rtdetr_{backbone or 'stock'}_{init}_v3"
    if dataset == "DIVA-HisDB":
        base = "instance_segmentation/rtdetr_bbox_unet/diva-hisdb/detection/rtdetr_hf"
        rel = base if subset == "CB55" else f"{base}/{subset}"
        coco = f"00_data/DIVA-HisDB/coco_task2_{subset}"
    else:
        rel = f"instance_segmentation/rtdetr_bbox_unet/u-diads-tl/detection/rtdetr_hf/{subset}"
        coco = ""                        # U-DIADS masks are already the target
    env = dict(DATASET=subset, BACKBONE=backbone, INIT=init,
               IMAGE_SIZE=str(IMAGE_SIZE), RUN_NAME=name,
               NUM_EPOCHS=str(EPOCHS), USE_EMA="0")
    if coco:
        env["COCO_DIR"] = coco
    return name, rel, env


failed = []
for i, (dataset, subset, backbone, init) in enumerate(ARMS, 1):
    name, rel, env = spec(dataset, subset, backbone, init)
    if (LOCAL_MODELS / rel / name / "best_model" / "config.json").exists():
        print(f"[skip] ({i}/{len(ARMS)}) {subset}/{name}")
        continue

    print(f"\n######## ({i}/{len(ARMS)}) {dataset} {subset} / {name} ########")
    if sh("python train_rtdetr_hf.py", check=False, cwd=STAGE_DIR, env=env).returncode:
        print(f"[FAILED] {subset}/{name}")
        failed.append(f"{subset}/{name}")
        continue
    src = LOCAL_MODELS / rel / name
    if src.is_dir():
        shutil.copytree(src, DRIVE_MODELS / rel / name, dirs_exist_ok=True)
        print(f"[push] {rel}/{name}")

print(f"\nготово: {len(ARMS) - len(failed)}/{len(ARMS)}")
if failed:
    print("не обучились:", failed)

## 8. Выгрузка

Детекторные метрики пишутся в `results.csv` каждого подмножества. Линейные метрики
считаются локально — там уже настроены сегментер кропов, Java-оценщик для DIVA
и реализация Zottin для U-DIADS.

In [ ]:

for rel in [f"instance_segmentation/rtdetr_bbox_unet/diva-hisdb/rtdetr_hf{'' if s == 'CB55' else '/' + s}" for s in DIVA] \
         + [f"instance_segmentation/rtdetr_bbox_unet/u-diads-tl/rtdetr_hf/{s}" for s in UDIADS]:
    src = REPO_DIR / "99_evaluation" / rel / "results.csv"
    if src.exists():
        dst = DRIVE_OUT / rel / "results.csv"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"[push] {rel}/results.csv")

import pandas as pd
for rel in [f"instance_segmentation/rtdetr_bbox_unet/diva-hisdb/rtdetr_hf{'' if s == 'CB55' else '/' + s}" for s in DIVA] \
         + [f"instance_segmentation/rtdetr_bbox_unet/u-diads-tl/rtdetr_hf/{s}" for s in UDIADS]:
    p = REPO_DIR / "99_evaluation" / rel / "results.csv"
    if not p.exists():
        continue
    d = pd.read_csv(p)
    d = d[d.platform.str.endswith("_v3", na=False)]
    if len(d):
        print(f"\n{rel}: {len(d)} арок, медиана test_mAP50 {d.test_mAP50.median():.3f}")
        print(d[["platform", "test_mAP50"]].sort_values("test_mAP50", ascending=False)
              .head(3).to_string(index=False))

## 9. Дальше — локально

1. Скачайте `80_models/instance_segmentation/rtdetr_bbox_unet/` с Диска в локальный репозиторий.
2. Посчитайте линейные метрики. **Обязательно с `close-fraction=0`** — значение по
   умолчанию 0.04 сваривает соседние строки и стоит 0.24 FM на Syr341:
   ```
   python 50_modelling/instance_segmentation/rtdetr_bbox_unet/evaluate_loss_ablation_detr_udiads.py \
       --subset Syr341 --detectors <run> --arms tversky --close-fraction 0.0 \
       --dedup-iou 0.5 --segmenter-threshold 0.9
   ```
   Для DIVA — `predict_and_eval_rtdetr_diva.py`, там подбор двумерный и
   автоматический.
3. Пересоберите таблицы: `python 99_evaluation/scripts/make_rq1_split_tables.py`.